# Stage 4 – Explainable AI

Global and local explanations of the XGBoost model with SHAP (summary, dependence plots) and LIME.

Part of **Explainable AI for Credit Risk Prediction**. See the main README for the full pipeline.

In [1]:
# 04_xai_credit_risk.ipynb
# =========================================
# Stage 4: Explainable AI & Model Interpretation
# Project: Explainable AI for Credit Risk Prediction(sai)
# Uses:
#   - SHAP for global + local explanations
#   - LIME for local, human-readable explanations
# =========================================

import os
import gc
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import shap
from lime.lime_tabular import LimeTabularExplainer

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    precision_recall_curve,
    auc,
)

plt.style.use("seaborn-v0_8-whitegrid")

# -----------------------------------------------------
# 0. Config & Paths
# -----------------------------------------------------
SAVE_DIR = "../outputs/notebook3_outputs"   # same directory used in Notebook 3
os.makedirs(SAVE_DIR, exist_ok=True)

CLEANED_SAMPLE_PATH = os.path.join(SAVE_DIR, "cleaned_sample_for_modelling.csv.gz")
META_PATH           = os.path.join(SAVE_DIR, "feature_metadata.joblib")
XGB_PATH            = os.path.join(SAVE_DIR, "xgboost.joblib")
SHAP_EXPL_PATH      = os.path.join(SAVE_DIR, "shap_explainer.joblib")  # optional

TARGET_COL = "default"   # target column created in Notebook 1/3

print("Using paths:")
print("  Cleaned sample:", CLEANED_SAMPLE_PATH)
print("  Metadata:      ", META_PATH)
print("  XGBoost model: ", XGB_PATH)
print("  SHAP explainer:", SHAP_EXPL_PATH)

# -----------------------------------------------------
# 1. Load Cleaned Sample & Metadata
# -----------------------------------------------------
if not os.path.exists(CLEANED_SAMPLE_PATH):
    raise FileNotFoundError(
        f"Cleaned sample not found at {CLEANED_SAMPLE_PATH}. "
        f"Run Notebook 3 first."
    )

cleaned_sample = pd.read_csv(CLEANED_SAMPLE_PATH, compression="infer")
if TARGET_COL not in cleaned_sample.columns:
    raise ValueError(
        f"Target column '{TARGET_COL}' not found in cleaned sample. "
        f"Check Notebook 3 output / target name."
    )

print("\n✅ Loaded cleaned sample for modelling.")
print("Shape:", cleaned_sample.shape)

# Separate features and target
y = cleaned_sample[TARGET_COL].astype(int)
X = cleaned_sample.drop(columns=[TARGET_COL])

print("Features:", X.shape, "| Target:", y.shape)

# Load metadata (feature names etc.)
if not os.path.exists(META_PATH):
    raise FileNotFoundError(
        f"Metadata file not found at {META_PATH}. Run Notebook 3 first."
    )

meta = joblib.load(META_PATH)
feature_names  = meta.get("feature_names", list(X.columns))
numeric_cols   = meta.get("numeric_cols", [])
other_cols     = meta.get("other_cols", [])
large_cat_used = meta.get("large_cat_used", [])
small_cat_used = meta.get("small_cat_used", [])

print("\n✅ Metadata loaded.")
print(f"Number of features in metadata: {len(feature_names)}")
print("Example feature names:", feature_names[:10])

# Ensure X column order matches feature_names, if possible
if len(feature_names) == X.shape[1]:
    X = X[feature_names]
else:
    print("\n⚠️ Warning: feature_names length does not match X columns.")
    print("Using X columns as-is; SHAP/LIME will still work but order is X.columns.")

# Convert to NumPy for SHAP & LIME
X_np = X.to_numpy(dtype=float)

# -----------------------------------------------------
# 2. Load Trained XGBoost Model
# -----------------------------------------------------
if not os.path.exists(XGB_PATH):
    raise FileNotFoundError(
        f"XGBoost model not found at {XGB_PATH}. Run Notebook 3 first."
    )

xgb_clf = joblib.load(XGB_PATH)
print("\n✅ XGBoost model loaded.")

# Quick sanity check (should roughly match Notebook 3 metrics)
probs = xgb_clf.predict_proba(X_np)[:, 1]
y_pred = (probs >= 0.5).astype(int)

print("\n=== Sanity Check: XGBoost Performance on Saved Sample ===")
print(classification_report(y, y_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y, y_pred))

roc = roc_auc_score(y, probs)
prec, rec, _ = precision_recall_curve(y, probs)
pr_auc = auc(rec, prec)
print(f"ROC AUC: {roc:.6f}, PR AUC: {pr_auc:.6f}")

# -----------------------------------------------------
# 3. SHAP – Global Explanations
# -----------------------------------------------------
print("\n3) SHAP – building / loading explainer...")

# Try to load existing SHAP explainer, else build a new one
if os.path.exists(SHAP_EXPL_PATH):
    try:
        shap_bundle = joblib.load(SHAP_EXPL_PATH)
        explainer = shap_bundle.get("explainer", None)
        if explainer is not None:
            print("✅ Loaded existing SHAP explainer from disk.")
        else:
            raise ValueError("Explainer missing in bundle; rebuilding...")
    except Exception as e:
        print(f"⚠️ Failed to load SHAP explainer ({e}). Rebuilding...")
        explainer = shap.TreeExplainer(xgb_clf)
else:
    print("No saved SHAP explainer found. Building a new one...")
    explainer = shap.TreeExplainer(xgb_clf)

# Sample subset for SHAP (for speed)
shap_sample_size = min(2000, X_np.shape[0])
rng = np.random.RandomState(42)
shap_indices = rng.choice(np.arange(X_np.shape[0]), size=shap_sample_size, replace=False)

X_shap = X_np[shap_indices]
y_shap = y.iloc[shap_indices].values

print(f"\nComputing SHAP values on subset of size {shap_sample_size}...")
shap_values = explainer.shap_values(X_shap)

# For xgboost binary classification, shap_values can be:
# - a 2D array (n_samples, n_features), or
# - a list [values_for_class0, values_for_class1]
if isinstance(shap_values, list):
    # Class 1 (default) explanations
    shap_vals_for_plot = shap_values[1]
else:
    shap_vals_for_plot = shap_values

# Global mean absolute SHAP
mean_abs_shap = np.abs(shap_vals_for_plot).mean(axis=0)
shap_importance = sorted(
    zip(feature_names, mean_abs_shap),
    key=lambda x: x[1],
    reverse=True
)

print("\nTop 20 features by mean |SHAP|:")
for feat, imp in shap_importance[:20]:
    print(f"  {feat}: {imp:.6f}")

# SHAP summary bar plot (global importance)
print("\nSaving SHAP summary plots...")
plt.figure()
shap.summary_plot(
    shap_vals_for_plot,
    X_shap,
    feature_names=feature_names,
    plot_type="bar",
    show=False
)
plt.tight_layout()
shap_bar_path = os.path.join(SAVE_DIR, "shap_summary_bar.png")
plt.savefig(shap_bar_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved SHAP bar summary to {shap_bar_path}")

# SHAP dot plot (impact per feature across samples)
plt.figure()
shap.summary_plot(
    shap_vals_for_plot,
    X_shap,
    feature_names=feature_names,
    show=False
)
plt.tight_layout()
shap_dot_path = os.path.join(SAVE_DIR, "shap_summary_dot.png")
plt.savefig(shap_dot_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved SHAP dot summary to {shap_dot_path}")

# Optional: SHAP dependence plot for top N features
TOP_N_FOR_DEP = 3
for feat, _ in shap_importance[:TOP_N_FOR_DEP]:
    if feat not in feature_names:
        continue
    feat_idx = feature_names.index(feat)
    plt.figure()
    shap.dependence_plot(
        feat_idx,
        shap_vals_for_plot,
        X_shap,
        feature_names=feature_names,
        show=False
    )
    plt.tight_layout()
    dep_path = os.path.join(SAVE_DIR, f"shap_dependence_{feat}.png")
    plt.savefig(dep_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved SHAP dependence plot for {feat} to {dep_path}")

# Optionally re-save explainer for reuse
joblib.dump({"explainer": explainer, "feature_names": feature_names}, SHAP_EXPL_PATH)
gc.collect()

# -----------------------------------------------------
# 4. LIME – Local Explanations
# -----------------------------------------------------
print("\n4) LIME – setting up local explainer for XGBoost...")

lime_train = X_np   # or X_np[:20000] if you want faster LIME
class_names = ["non_default", "default"]

lime_explainer = LimeTabularExplainer(
    training_data=lime_train,
    feature_names=feature_names,
    class_names=class_names,
    mode="classification",
    discretize_continuous=True
)

print("✅ LIME explainer initialised.")


def explain_with_lime(instance_idx: int, prefer_label: int = 1, save_plot: bool = True, suffix: str = ""):
    """
    Explain a single borrower with LIME.

    instance_idx : index in X_np / y
    prefer_label : class we would LIKE to explain (1 = default),
                   but we will fall back to an available label if needed.
    save_plot    : whether to save PNG
    suffix       : tag in filename, e.g. '_default' or '_nondefault'
    """
    if instance_idx < 0 or instance_idx >= X_np.shape[0]:
        raise IndexError(f"instance_idx {instance_idx} out of range [0, {X_np.shape[0]-1}]")

    # Feature vector for this borrower
    x_instance = X_np[instance_idx]

    # Model prediction
    probs = xgb_clf.predict_proba(x_instance.reshape(1, -1))[0]
    pred_label = int(np.argmax(probs))
    true_label = int(y.iloc[instance_idx])

    print("\n===================================")
    print(f"LIME explanation for test index {instance_idx}")
    print(f"True label:      {true_label} (0 = non-default, 1 = default)")
    print(f"Predicted probs: {probs}  (class 0, class 1)")
    print(f"Predicted class: {pred_label}")

    # Ask LIME for explanations (top_labels=2 → usually labels 0 and 1)
    lime_exp = lime_explainer.explain_instance(
        data_row=x_instance,
        predict_fn=xgb_clf.predict_proba,
        num_features=10,
        top_labels=2
    )

    available_labels = list(lime_exp.local_exp.keys())

    # Choose which label to use for explanation:
    # 1) try preferred label (default = 1), else
    # 2) predicted label, else
    # 3) first available in local_exp
    if prefer_label in available_labels:
        label_to_use = prefer_label
    elif pred_label in available_labels:
        label_to_use = pred_label
    else:
        label_to_use = available_labels[0]

    print(f"\nUsing label {label_to_use} for LIME explanation.")
    print("\nTop contributing features:")
    for feature, weight in lime_exp.as_list(label=label_to_use):
        print(f"  {feature}: {weight:.4f}")

    if save_plot:
        fig = lime_exp.as_pyplot_figure(label=label_to_use)
        plt.tight_layout()
        out_path = os.path.join(SAVE_DIR, f"lime_instance_{instance_idx}{suffix}.png")
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"LIME plot saved to {out_path}")

    return lime_exp


# -----------------------------------------------------
# 5. Run LIME on one default and one non-default borrower
# -----------------------------------------------------
default_indices    = np.where(y.values == 1)[0]
nondefault_indices = np.where(y.values == 0)[0]

if len(default_indices) > 0 and len(nondefault_indices) > 0:
    idx_default = int(default_indices[0])
    idx_nondeft = int(nondefault_indices[0])

    print("\nExplaining one DEFAULT borrower with LIME...")
    exp_def = explain_with_lime(idx_default,  prefer_label=1, save_plot=True, suffix="_default")

    print("\nExplaining one NON-DEFAULT borrower with LIME...")
    exp_non = explain_with_lime(idx_nondeft, prefer_label=1, save_plot=True, suffix="_nondefault")
else:
    print("\n⚠️ Could not find both classes in this sample. Skipping dual LIME examples.")

print("\n✅ Notebook 4 (XAI – SHAP + LIME) completed.")
print("All plots saved in:", SAVE_DIR)


d:\anaconda\envs\credit-risk-xai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using paths:
  Cleaned sample: ../outputs/notebook3_outputs\cleaned_sample_for_modelling.csv.gz
  Metadata:       ../outputs/notebook3_outputs\feature_metadata.joblib
  XGBoost model:  ../outputs/notebook3_outputs\xgboost.joblib
  SHAP explainer: ../outputs/notebook3_outputs\shap_explainer.joblib

✅ Loaded cleaned sample for modelling.
Shape: (40000, 86)
Features: (40000, 85) | Target: (40000,)

✅ Metadata loaded.
Number of features in metadata: 85
Example feature names: ['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'int_rate', 'installment', 'annual_inc', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high']

✅ XGBoost model loaded.

=== Sanity Check: XGBoost Performance on Saved Sample ===
              precision    recall  f1-score   support

           0     0.8131    0.9834    0.8902     32008
           1     0.5876    0.0948    0.1633      7992

    accuracy                         0.8058     40000
   macro avg     0.7004    0.5391    0.5268     40000
weighted avg     0.7

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>